# 2. Can a model pick the good crossovers?

Notebook 1 showed that only about 38 percent of golden cross trades make money. The original project tried to predict which ones. This notebook repeats that attempt with the three flaws removed:

* **The label is the outcome only.** The original label also counted a crossover as good when MACD and RSI looked strong, and those were inputs too.
* **Training uses the past only.** To score the crossovers of a year, the model sees only crossovers whose result was already known before that year began.
* **Inputs are ratios**, never raw prices.

In [ ]:
import os, sys, json
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, "src")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use("dark_background")
plt.rcParams.update({"figure.facecolor": "black", "axes.facecolor": "black", "axes.edgecolor": "#3a3b3e", "axes.grid": True,
                     "grid.color": "#1c1c1e", "figure.figsize": (11, 4.5), "axes.spines.top": False, "axes.spines.right": False})
WHITE, SILVER, GREY, JADE = "#f2f2f0", "#d4d6da", "#8d8d90", "#5fd0a0"
pd.options.display.float_format = "{:.3f}".format

## One row per crossover

Each row holds what was known at the close on the day of the cross, and what happened afterward measured from the next open.

In [ ]:
from emalab.universe import load_prices, membership, START
from emalab.features import crossover_events, FEATURES

wide, spy, info = load_prices()
member = membership(wide["close"].index, list(wide["close"].columns), info)
pieces = []
for ticker in wide["close"].columns:
    frame = pd.DataFrame({field: wide[field][ticker] for field in wide}).dropna(subset=["close"])
    if len(frame) < 400:
        continue
    frame["open"] = frame["open"].fillna(frame["close"])
    events = crossover_events(frame, spy)
    if len(events):
        events = events[events["date"].map(lambda d: bool(member.at[d, ticker])) & (events["date"] >= START)]
        pieces.append(events.assign(ticker=ticker))
events = pd.concat(pieces, ignore_index=True)
events["trade_made_money"] = events["trade_result"] > 0
events["up_3_in_10_days"] = events["forward_10"] > 0.03
print(f"{len(events):,} crossovers from {events.ticker.nunique()} stocks, {events.date.min():%Y} to {events.date.max():%Y}")
print(f"Share whose trade made money: {events.trade_made_money[events.trade_result.notna()].mean():.0%}")
print(f"Share up more than 3% after 10 days: {events.up_3_in_10_days[events.forward_10.notna()].mean():.0%}")
events[["date", "ticker"] + FEATURES[:5] + ["forward_10", "trade_result"]].tail()

## The walk forward test

Two questions are tested with the same method.

1. **Did the trade make money?** This is what a scanner needs to know.
2. **Was the stock up more than 3 percent after 10 days?** This is the original project's target, without the MACD and RSI shortcut.

The score to watch is ranking skill: 0.50 is a coin flip and 1.00 is perfect. The original project reported 0.89.

In [ ]:
from emalab.model import walk_forward, skill, compare_groups

events["known_10"] = events["date"] + pd.Timedelta(days=21)      # 10 trading days is at most 3 weeks on the calendar
trade_test = walk_forward(events, "trade_made_money", "known_on")
ten_day_test = walk_forward(events, "up_3_in_10_days", "known_10")
print(f"Did the trade make money?   skill {skill(trade_test, 'trade_made_money'):.3f}  on {len(trade_test):,} crossovers from 2010 on")
print(f"Up 3% after 10 days?        skill {skill(ten_day_test, 'up_3_in_10_days'):.3f}  on {len(ten_day_test):,} crossovers")

## Is the test itself honest?

A check on the method: scramble the training answers and run it again. A sound test must then fall to a coin flip. If it did not, something would be leaking.

In [ ]:
scrambled = walk_forward(events, "trade_made_money", "known_on", shuffle=True)
print(f"With scrambled answers: skill {skill(scrambled, 'trade_made_money'):.3f}  (should be close to 0.50)")

## Skill year by year

A real edge should show up in most years, not just on average.

In [ ]:
from sklearn.metrics import roc_auc_score
by_year = trade_test.groupby(trade_test.date.dt.year).apply(lambda g: roc_auc_score(g.trade_made_money, g.score) if g.trade_made_money.nunique() > 1 else np.nan)
fig, ax = plt.subplots()
ax.bar(by_year.index, by_year - 0.5, bottom=0.5, color=[JADE if v > 0.5 else GREY for v in by_year])
ax.axhline(0.5, color=SILVER, lw=1); ax.set_ylabel("Ranking skill (0.50 is a coin flip)"); ax.set_ylim(0.35, 0.75)
plt.show()
print(f"Years above a coin flip: {(by_year > 0.5).sum()} of {by_year.notna().sum()}")

## Does picking help where it counts?

Skill scores are abstract. This is the practical version: take only the crossovers the model ranked in its top third at the time, and compare them with the ones it skipped.

In [ ]:
groups = compare_groups(trade_test)
groups.style.format({"crossovers": "{:,.0f}", "share that made money": "{:.0%}", "average trade": "{:+.1%}", "typical trade": "{:+.1%}", "average next 10 days": "{:+.2%}"})

In [ ]:
fig, ax = plt.subplots()
ordered = trade_test.sort_values("date")
for mask, color, label in [(ordered.picked, JADE, "Picked by the model"), (~ordered.picked, GREY, "Skipped by the model")]:
    part = ordered[mask]
    ax.plot(part["date"], part["trade_result"].expanding().mean(), color=color, lw=1.5, label=label)
ax.axhline(ordered["trade_result"].mean(), color=SILVER, lw=0.8, ls=":")
ax.yaxis.set_major_formatter(lambda v, _: f"{v:.0%}"); ax.set_ylabel("Average trade result so far"); ax.legend(frameon=False)
plt.show()

## What the model leans on

Which inputs matter most, measured by how much the ranking skill falls when each one is scrambled. Trained on everything before 2020 and measured on 2020 onward.

In [ ]:
from sklearn.inspection import permutation_importance
from emalab.model import new_model
usable = events.dropna(subset=["trade_result", "known_on"])
past, later = usable[usable.known_on < "2020-01-01"], usable[usable.date >= "2020-01-01"]
fitted = new_model().fit(past[FEATURES], past.trade_made_money)
weight = permutation_importance(fitted, later[FEATURES], later.trade_made_money, scoring="roc_auc", n_repeats=8, random_state=0)
order = np.argsort(weight.importances_mean)
fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(np.array(FEATURES)[order], weight.importances_mean[order], color=SILVER)
ax.set_xlabel("Drop in ranking skill when scrambled")
plt.show()

In [ ]:
trade_test.to_csv("results/model_scored_crossovers.csv", index=False)
summary = {"crossovers_scored": len(trade_test), "skill_trade_made_money": skill(trade_test, "trade_made_money"),
           "skill_up_3_in_10_days": skill(ten_day_test, "up_3_in_10_days"), "skill_with_scrambled_answers": skill(scrambled, "trade_made_money"),
           "years_above_coin_flip": int((by_year > 0.5).sum()), "years_tested": int(by_year.notna().sum()),
           "groups": json.loads(groups.reset_index().to_json(orient="records")),
           "top_inputs": [FEATURES[i] for i in order[::-1][:5]]}
json.dump(summary, open("results/model_test.json", "w"), indent=1)
summary